<img src="images/Exercise_merge.png" width="1000"/>

In [1]:
import heapq

def merge(*l):

    return heapq.merge(*l)

In [7]:
a=[1, 3, 5, 7]
b=[2, 4, 6, 8]

for i in merge(a,b):
    print(i, end=' ')

print('/n')
print(a)
print(b)


1 2 3 4 5 6 7 8 /n
[1, 3, 5, 7]
[2, 4, 6, 8]


## Yes — Python Has `heapq.merge()`

Python's `heapq` module has a built-in function specifically designed to merge sorted lists **without re-sorting**.

---

### The Solution — `heapq.merge()`

```python
import heapq

a = [1, 3, 5, 7]
b = [2, 4, 6, 8]

merged = list(heapq.merge(a, b))
print(merged)   # → [1, 2, 3, 4, 5, 6, 7, 8]
```

It intelligently interleaves the two sorted lists by comparing elements as it goes — no sorting happens.

---

### How it works internally

`heapq.merge()` uses a technique called the **merge step from merge sort**:

1. Look at the first element of both lists
2. Take the smaller one and add it to the result
3. Move to the next element in that list
4. Repeat until both lists are exhausted

```
a = [1, 3, 5]     b = [2, 4, 6]

Compare 1 vs 2 → take 1     result = [1]
Compare 3 vs 2 → take 2     result = [1, 2]
Compare 3 vs 4 → take 3     result = [1, 2, 3]
Compare 5 vs 4 → take 4     result = [1, 2, 3, 4]
Compare 5 vs 6 → take 5     result = [1, 2, 3, 4, 5]
Only 6 left    → take 6     result = [1, 2, 3, 4, 5, 6]
```

This is **O(n + m)** — very efficient. Actual sorting would be **O((n+m) log(n+m))**.

---

### Cool features of `heapq.merge()`

**1. Returns a generator (lazy)**
It doesn't build the whole list at once — it yields values as you iterate:
```python
for x in heapq.merge([1, 3, 5], [2, 4, 6]):
    print(x)   # → 1, 2, 3, 4, 5, 6 (one at a time)
```

**2. Can merge multiple lists at once**
```python
a = [1, 4, 7]
b = [2, 5, 8]
c = [3, 6, 9]

merged = list(heapq.merge(a, b, c))
print(merged)   # → [1, 2, 3, 4, 5, 6, 7, 8, 9]
```

**3. Supports descending merge with `reverse=True`**
```python
a = [7, 5, 3, 1]
b = [8, 6, 4, 2]
list(heapq.merge(a, b, reverse=True))
# → [8, 7, 6, 5, 4, 3, 2, 1]
```

**4. Supports custom keys**
```python
a = [("apple", 3), ("cherry", 1)]
b = [("banana", 2), ("date", 4)]

list(heapq.merge(a, b, key=lambda x: x[1]))
# merges based on the second element of each tuple
```

---

### Writing it yourself (for learning)

If you want to understand the mechanic, here's a manual version:

```python
def merge_sorted(a, b):
    result = []
    i = j = 0
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            result.append(a[i])
            i += 1
        else:
            result.append(b[j])
            j += 1
    # Add whatever is left
    result.extend(a[i:])
    result.extend(b[j:])
    return result

print(merge_sorted([1, 3, 5], [2, 4, 6]))
# → [1, 2, 3, 4, 5, 6]
```

Same logic as `heapq.merge()` — just written explicitly.

---

### Comparison of approaches

| Approach | Speed | Memory | Sorted lists? |
|---|---|---|---|
| `heapq.merge(a, b)` | O(n+m) | Lazy (generator) | **Required** |
| `sorted(a + b)` | O(k log k) | Full list | Any lists |
| Manual merge | O(n+m) | Full list | **Required** |

---

### When to use which

**Use `heapq.merge()` when:**
- Both inputs are **already sorted**
- You want **maximum efficiency**
- You want to work with a stream (generator) rather than a full list

**Use `sorted(a + b)` when:**
- Inputs might **not be sorted** already
- Simplicity matters more than performance
- Lists are small

---

### Simple analogy

> Imagine two lines of people at a bank, each already sorted by ticket number 🎫:
> - **`heapq.merge`** = a smart teller who peeks at both lines and calls whoever has the next lowest number. Fast and simple.
> - **`sorted(a + b)`** = combining both lines into one big group, then sorting everyone again from scratch. Wasteful when the two lines were already sorted.

---

### The key takeaway
> `heapq.merge()` is Python's built-in tool for **merging pre-sorted sequences without sorting again**. It's fast (O(n+m)), memory-efficient (lazy), and handles multiple lists and custom keys. Perfect for your use case.

## What "Passed by Reference" Means

When you pass a list to a function, Python does **not** make a copy of it. Instead, both the caller and the function share the **same list**. So if the function changes the list, the caller sees the change too.

---

### The Basic Idea

```python
def add_item(my_list):
    my_list.append(99)      # modifies the list

L = [1, 2, 3]
add_item(L)
print(L)                    # → [1, 2, 3, 99]   ← the original changed!
```

Even though we only "passed" `L` to the function, the function's changes **affect the original list** in the caller's scope.

---

### Why does this happen?

Python doesn't send a **copy** of the list — it sends a **reference** (like a pointer / label) to the same list object in memory:

```
Caller side:   L ────────┐
                          ↓
                      [1, 2, 3, 99]     ← one list in memory
                          ↑
Function:      my_list ──┘
```

Both `L` and `my_list` point to the **same object**. Any change through one is visible to the other.

---

### The Danger

Consider a well-meaning function that accidentally modifies its input:

```python
def sort_and_print(items):
    items.sort()           # sorts in place — modifies caller's list!
    print(items)

data = [3, 1, 2]
sort_and_print(data)
print(data)                # → [1, 2, 3]  ← original also sorted!
```

The caller probably didn't expect `data` to change — but it did. This is a **hidden side effect**, and it can cause bugs that are hard to track down.

---

### How to Avoid This — Two Common Patterns

#### Pattern 1 — Make a copy inside the function
```python
def sort_and_print(items):
    items = items.copy()   # work on a copy
    items.sort()
    print(items)

data = [3, 1, 2]
sort_and_print(data)
print(data)                # → [3, 1, 2]  ← unchanged ✓
```

Other ways to copy a list:
```python
items[:]         # slice copy
list(items)      # constructor copy
items.copy()     # method copy
```

#### Pattern 2 — Return a new list instead of modifying
```python
def get_sorted(items):
    return sorted(items)   # sorted() returns a new list

data = [3, 1, 2]
new = get_sorted(data)
print(data)   # → [3, 1, 2]  ← unchanged
print(new)    # → [1, 2, 3]
```

Notice the difference:
- `items.sort()` → **modifies in place** (mutating)
- `sorted(items)` → **returns a new list** (non-mutating)

---

### When It Does NOT Happen — Immutable Types

For **immutable** types (int, float, str, tuple), you don't have this problem, because they **can't be modified anyway**:

```python
def change(x):
    x = x + 1
    print(x)     # → 6

n = 5
change(n)
print(n)         # → 5   ← unchanged ✓
```

Even though `n` is passed to the function, integers can't be mutated, so no danger exists.

---

### The Contrast Table

| Type | Passed by reference? | Can function affect caller? |
|---|---|---|
| `list` | ✓ Yes | ✓ Yes (dangerous!) |
| `dict` | ✓ Yes | ✓ Yes |
| `set` | ✓ Yes | ✓ Yes |
| `int`, `float` | ✓ Yes (but immutable) | ✗ No |
| `str` | ✓ Yes (but immutable) | ✗ No |
| `tuple` | ✓ Yes (but immutable) | ✗ No |

**Rule of thumb:** if the type is **mutable** (list, dict, set), you can accidentally modify the caller's data.

---

### Simple Analogy

> Think of a list as a **shared Google Doc** 📄
> - You share the **link** with someone (that's the "reference")
> - You did NOT send them a copy of the document
> - If they edit the doc, **you see the changes too**
>
> Now imagine sending them the link and asking them to *"just have a look"* — but they end up making edits. Same thing happens with Python lists in functions.
>
> To be safe:
> - **Make them a copy** first (`items.copy()`)
> - Or **ask for a new document back** (return a new list)

---

### The Key Takeaway

> When a function receives a **mutable** object (like a list), it can silently modify the caller's data. To avoid unexpected changes:
>
> 1. **Copy** the input inside the function if you need to modify it, OR
> 2. **Return a new** object instead of modifying in place, OR
> 3. **Document clearly** that your function modifies its input (like `list.sort()` does)
>
> When writing functions, ask yourself: *"Do I really want to modify the caller's data?"* — if not, work on a copy.